# Update log
2024/08/29
- Test without Responsivity and/or baseline
- Test with reduced number of features
- Baseline alone gives up to 75% accuracy
- Suspect data distribution due to always running experiment in 0,1,2,3,4
- Run experiment in De Bruijn sequence to balance the adjacent experiment channels
---

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np
import json

debruijn_1 = "D:\\code\\uom_explore\\processed_data\\metrics_de_brujin_large_even_1.csv"


spk_json = "D:\\code\\uom_explore\\data_science\\scripts\\parameter.json"

data_path = debruijn_1
param_path = spk_json

with open('parameter.json','r') as file:
    params = json.load(file)

# Hyperparameters
# Extract parameters from the JSON object
# hidden_size = params['hidden_size']
ground_truth = params['ground_truth']
num_epochs = params['mlp']['num_epochs']
batch_size = params['mlp']['batch_size']
learning_rate = params['mlp']['learning_rate']
momentum_value = params['mlp']['momentum_value']
dropout_rate = params['mlp']['dropout']

df = pd.read_csv(data_path)
# drop experiment_id column
# df.drop('experiment_id', axis=1, inplace=True)
print(type(df))
df.head()



KeyError: 'num_epochs'

## Load all features

In [3]:
# Get all column names from the DataFrame
all_columns = df.columns.tolist()

# Remove 'channel_id' and the ground truth from the list of features
features = [col for col in all_columns if col != 'experiment_id' and col != ground_truth]

# Print the features
print("Features:")
print(json.dumps(features, indent=2))



Features:
[
  "baseline_140",
  "baseline_143",
  "baseline_146",
  "baseline_149",
  "baseline_152",
  "baseline_155",
  "baseline_158",
  "baseline_161",
  "baseline_164",
  "baseline_167",
  "baseline_170",
  "baseline_173",
  "baseline_176",
  "baseline_179",
  "baseline_182",
  "baseline_185",
  "baseline_188",
  "baseline_191",
  "baseline_194",
  "baseline_197",
  "baseline_200",
  "baseline_203",
  "baseline_206",
  "baseline_209",
  "baseline_212",
  "baseline_215",
  "baseline_218",
  "baseline_221",
  "baseline_224",
  "baseline_227",
  "baseline_230",
  "baseline_233",
  "baseline_236",
  "baseline_239",
  "baseline_242",
  "baseline_245",
  "baseline_248",
  "baseline_250",
  "max_reaction_R_140",
  "max_reaction_R_143",
  "max_reaction_R_146",
  "max_reaction_R_149",
  "max_reaction_R_152",
  "max_reaction_R_155",
  "max_reaction_R_158",
  "max_reaction_R_161",
  "max_reaction_R_164",
  "max_reaction_R_167",
  "max_reaction_R_170",
  "max_reaction_R_173",
  "max_reaction_

# Keep all features

In [4]:
# X includes all features
X = df[features]



## Select settings to keep

In [ ]:
# # Extract all unique numbers from feature names
# feature_numbers = set()
# for feature in features:
#     parts = feature.split('_')
#     if len(parts) > 1 and parts[-1].isdigit():
#         feature_numbers.add(int(parts[-1]))

# # Sort the numbers
# sorted_numbers = sorted(feature_numbers)

# # Select settings to keep
# # settings_to_keep = [140, 150, 152, 155, 157, 160, 162, 165, 167, 170, 172, 175, 177, 180, 182, 185, 187, 190, 192, 195]
# settings_to_keep = [140]

# # Filter the features to keep only the selected settings
# features_to_keep = [feature for feature in features if int(feature.split('_')[-1]) in settings_to_keep]

# # Update the features list
# features = features_to_keep

# # Print the features
# print("Features:")
# print(json.dumps(features, indent=2))


## Select features to keep

In [ ]:
# # Remove 'responsivity_' and 'baseline_' features from the dataset
# features_to_keep = [col for col in features if not (col.startswith('responsivity_'))]
# # features_to_keep = [col for col in features_to_keep if not (col.startswith('responsivity_'))]
# # Update the features list
# features = features_to_keep

# # Update X dataframe to only include the kept features
# X = df[features]

# # Update input_size
# input_size = len(features)

# print(f"Features after removing 'responsivity_':")
# print(json.dumps(features, indent=2))
# print(f"New input size: {input_size}")


# Data split and scale

In [5]:
# Preview features
X = X[features]
print(X.head())


   baseline_140  baseline_143  baseline_146  baseline_149  baseline_152  \
0   1304.321762    851.782734    822.821395    861.828490    915.179956   
1   1089.128873    686.398724    664.426060    698.958067    745.947882   
2    933.216271    584.428862    567.101092    598.706476    641.556892   
3    862.767043    544.731229    530.892613    561.774378    603.985331   
4   1193.223012    759.743885    752.950952    805.589437    869.654724   

   baseline_155  baseline_158  baseline_161  baseline_164  baseline_167  ...  \
0    974.803680   1036.697018   1097.286209   1153.358682   1204.909788  ...   
1    798.561828    855.173888    921.501706    980.874976   1036.374049  ...   
2    688.345655    740.002441    805.943269    865.494236    925.072213  ...   
3    649.670223    701.950674    766.896687    827.482118    885.206427  ...   
4    943.912449   1026.417458   1131.115262   1226.542268   1317.309053  ...   

   temperature_max  temperature_std  humidity_mean  humidity_min  \


In [6]:
input_size = len(X.columns)  # removing the ground truth from the number of columns counted
num_classes = df[ground_truth].nunique()
print(f"Number of classes: {num_classes}")
print(f"Number of features: {input_size}")

# Preview ground truth
y = df[ground_truth]

# Split into training, validation, and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42)  # This makes 60%, 20%, 20%

# Initialize the StandardScaler
scaler = StandardScaler()
# scaler = MinMaxScaler(feature_range=(0,255)) # 

# Fit the scaler to the training data and transform it
X_train_scaled = scaler.fit_transform(X_train)

# Apply the same transformation to validation and test sets
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Convert arrays to tensors
X_train_scaled = torch.tensor(X_train_scaled, dtype=torch.float32).unsqueeze(1)  # Shape: [batch_size, 1, num_features]
y_train = torch.tensor(y_train.to_numpy(), dtype=torch.long)  # Convert to NumPy array first
X_val_scaled = torch.tensor(X_val_scaled, dtype=torch.float32).unsqueeze(1)  # Shape: [batch_size, 1, num_features]
y_val = torch.tensor(y_val.to_numpy(), dtype=torch.long)  # Convert to NumPy array first
X_test_scaled = torch.tensor(X_test_scaled, dtype=torch.float32).unsqueeze(1)  # Shape: [batch_size, 1, num_features]
y_test = torch.tensor(y_test.to_numpy(), dtype=torch.long)  # Convert to NumPy array first

# Create datasets
train_dataset = TensorDataset(X_train_scaled, y_train)
val_dataset = TensorDataset(X_val_scaled, y_val)
test_dataset = TensorDataset(X_test_scaled, y_test)

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

Number of classes: 5
Number of features: 126


# 1DCNN

In [7]:
# Define the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

in_channels = params['cnn']['in_channels']
out_channels = params['cnn']['out_channels']
kernel_sizes = params['cnn']['kernel_sizes']

class CNN1DClassifier(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_sizes, num_classes, dropout_rate=0.5):
        super(CNN1DClassifier, self).__init__()
        
        assert len(out_channels) == len(kernel_sizes), "The length of out_channels and kernel_sizes must be the same"
        
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()  # Adding batch normalization if needed
        
        current_in_channels = in_channels
        
        for out_channel, kernel_size in zip(out_channels, kernel_sizes):
            self.convs.append(nn.Conv1d(current_in_channels, out_channel, kernel_size=kernel_size, stride=1, padding=kernel_size // 2))
            self.bns.append(nn.BatchNorm1d(out_channel))  # Optional: Add batch normalization
            current_in_channels = out_channel
        
        # Calculate the size after all convolutional and pooling layers
        conv_output_size = input_size
        for kernel_size in kernel_sizes:
            conv_output_size = (conv_output_size + 2 * (kernel_size // 2) - (kernel_size - 1) - 1) // 1 + 1
            conv_output_size = conv_output_size // 2  # After pooling
        
        self.pool = nn.MaxPool1d(kernel_size=2, stride=2, padding=0)
        self.fc1 = nn.Linear(out_channels[-1] * conv_output_size, 128)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(128, num_classes)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        # print(f'Input shape: {x.shape}')
        for conv, bn in zip(self.convs, self.bns):
            x = self.pool(torch.relu(bn(conv(x))))
            # print(f'After conv and pool: {x.shape}')
        x = x.view(x.size(0), -1)  # Flatten the tensor
        # print(f'After flatten: {x.shape}')
        x = torch.relu(self.fc1(x))
        # print(f'After fc1: {x.shape}')
        x = self.dropout(x)
        x = self.fc2(x)
        # print(f'After fc2: {x.shape}')
        x = self.softmax(x)
        return x

# Train the 1D CNN model

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=25, device='cpu'):
    model = model.to(device)
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = running_corrects.double() / len(train_loader.dataset)

        print(f'Epoch {epoch}/{num_epochs - 1}')
        print(f'Training Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

        # Validation phase
        model.eval()
        val_loss = 0.0
        val_corrects = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)
                val_corrects += torch.sum(preds == labels.data)

        val_loss = val_loss / len(val_loader.dataset)
        val_acc = val_corrects.double() / len(val_loader.dataset)

        print(f'Validation Loss: {val_loss:.4f} Acc: {val_acc:.4f}')

    return model

# Initialize model, loss function, and optimizer
model = CNN1DClassifier(in_channels, out_channels, kernel_sizes, num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Train the model
trained_model = train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=num_epochs)

# Evaluate the model on the test set
model.eval()
test_loss = 0.0
test_corrects = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        loss = criterion(outputs, labels)

        test_loss += loss.item() * inputs.size(0)
        test_corrects += torch.sum(preds == labels.data)

test_loss = test_loss / len(test_loader.dataset)
test_acc = test_corrects.double() / len(test_loader.dataset)

print(f'Test Loss: {test_loss:.4f} Acc: {test_acc:.4f}')


Epoch 0/149
Training Loss: 1.5997 Acc: 0.2613
Validation Loss: 1.6030 Acc: 0.2604
Epoch 1/149
Training Loss: 1.5076 Acc: 0.3902
Validation Loss: 1.5850 Acc: 0.5208
Epoch 2/149
Training Loss: 1.4212 Acc: 0.4808
Validation Loss: 1.5346 Acc: 0.4167
Epoch 3/149
Training Loss: 1.3313 Acc: 0.5993
Validation Loss: 1.4818 Acc: 0.3958
Epoch 4/149
Training Loss: 1.2573 Acc: 0.6899
Validation Loss: 1.4109 Acc: 0.5521
Epoch 5/149
Training Loss: 1.2222 Acc: 0.7247
Validation Loss: 1.3451 Acc: 0.5833
Epoch 6/149
Training Loss: 1.1854 Acc: 0.7526
Validation Loss: 1.2490 Acc: 0.6875
Epoch 7/149
Training Loss: 1.1509 Acc: 0.7631
Validation Loss: 1.2162 Acc: 0.6979
Epoch 8/149
Training Loss: 1.1993 Acc: 0.7038
Validation Loss: 1.2167 Acc: 0.6979
Epoch 9/149
Training Loss: 1.1704 Acc: 0.7422
Validation Loss: 1.2212 Acc: 0.7292
Epoch 10/149
Training Loss: 1.1544 Acc: 0.7596
Validation Loss: 1.1728 Acc: 0.7396
Epoch 11/149
Training Loss: 1.1632 Acc: 0.7526
Validation Loss: 1.1764 Acc: 0.7396
Epoch 12/149
T